# Team B — Week 5: Systems Analysis & Report Figures
**IBM HPML Uncertainty-Aware Inference Project — Systems Lead**

Takes all profiler and vLLM results (Weeks 1-4 Mistral-7B + Week 5 cross-model)
and generates the final analysis and figures for the report's *systems section*.

**Sections:**
1. Load all results (profiler + vLLM + calibration)  
2. Profiler summary table (all 15 configs across 3 model families)  
3. Compute-bound vs. memory-bound analysis per precision level  
4. vLLM throughput comparison — kernel and memory efficiency  
5. Memory efficiency analysis (VRAM savings vs. FP16 baseline)  
6. Combined profiler × calibration scatter (accuracy vs. efficiency)  
7. Report-section narrative synthesis


## 0 · Setup

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── Drive paths ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT        = Path("/content/drive/MyDrive/uai_results")
MISTRAL_PROF_DIR  = DRIVE_ROOT / "TeamB"  / "profiler_results"
MISTRAL_VLLM_DIR  = DRIVE_ROOT / "TeamB"  / "vllm_results"
MISTRAL_CAL_DIR   = DRIVE_ROOT / "TeamB"  / "calibration_results"
LLAMA7B_PROF_DIR  = DRIVE_ROOT / "TeamA"  / "profiler_results"
LLAMA13B_PROF_DIR = DRIVE_ROOT / "TeamC"  / "profiler_results"
WEEK5_DIR         = DRIVE_ROOT / "week5"  / "cross_model_profiling"
OUT_DIR           = DRIVE_ROOT / "week5"  / "report_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── A100 hardware constants ─────────────────────────────────────────────────
A100_FP16_TFLOPS    = 312.0
A100_MEM_BW_TB_S    = 2.0
A100_RIDGE          = (A100_FP16_TFLOPS * 1e12) / (A100_MEM_BW_TB_S * 1e12)  # 156 FLOPs/byte

def classify_bound(ai: float) -> str:
    if ai == 0:              return "Unknown"
    if ai > A100_RIDGE*1.1:  return "Compute-bound"
    if ai < A100_RIDGE*0.9:  return "Memory-bound"
    return "Near ridge"


## 1 · Load All Results

In [ ]:
# ── 1a. Profiler results ───────────────────────────────────────────────────
PROF_DIRS = {
    "mistral-7b":  MISTRAL_PROF_DIR,
    "llama2-7b":   LLAMA7B_PROF_DIR,
    "llama2-13b":  LLAMA13B_PROF_DIR,
}

prof_results = {}
for family, dir_ in PROF_DIRS.items():
    if not dir_.exists():
        print(f"  ⚠  {dir_} not found — skipping {family}")
        continue
    for json_path in sorted(dir_.glob("*_profile.json")):
        config_key = json_path.stem.replace("_profile","")
        with open(json_path) as f:
            r = json.load(f)
        r.setdefault("family", family)
        prof_results[config_key] = r
        print(f"  Loaded {config_key}")

print(f"\nTotal profiler configs loaded: {len(prof_results)}")


In [ ]:
# ── 1b. vLLM throughput results (Mistral-7B from Week 4) ──────────────────
vllm_results = {}
if MISTRAL_VLLM_DIR.exists():
    for json_path in sorted(MISTRAL_VLLM_DIR.glob("*_vllm.json")):
        config_key = json_path.stem.replace("_vllm","")
        with open(json_path) as f:
            vllm_results[config_key] = json.load(f)
        print(f"  vLLM loaded: {config_key}")
else:
    print("  vLLM dir not found — vLLM section will be skipped")

print(f"Total vLLM results: {len(vllm_results)}")


In [ ]:
# ── 1c. Calibration results (Mistral-7B from Weeks 2-3) ──────────────────
cal_results = {}   # {config_key: {dataset: metrics_dict}}

if MISTRAL_CAL_DIR.exists():
    for config_dir in sorted(MISTRAL_CAL_DIR.iterdir()):
        if not config_dir.is_dir(): continue
        config_key = config_dir.name
        cal_results[config_key] = {}
        for json_path in sorted(config_dir.glob("*.json")):
            with open(json_path) as f:
                d = json.load(f)
            dataset = d.get("dataset", json_path.stem)
            cal_results[config_key][dataset] = d
        print(f"  Cal loaded: {config_key} — {list(cal_results[config_key])}")

print(f"Total cal configs: {len(cal_results)}")


In [ ]:
# ── 1d. Build unified DataFrame ────────────────────────────────────────────
rows = []
for key, r in prof_results.items():
    ai   = r["compute"]["arithmetic_intensity"]
    bits = r.get("bits", r["bits"]) if "bits" in r else None
    rows.append({
        "config_key":    key,
        "family":        r.get("family","?"),
        "description":   r.get("description", key),
        "quant_type":    r["quant_type"],
        "bits":          r["bits"],
        "infer_ms":      r["timing"]["total_inference_ms"],
        "tok_per_sec":   r["timing"]["tokens_per_second"],
        "peak_gpu_gb":   r["memory"]["peak_gpu_gb"],
        "avg_cuda_ms":   r["compute"]["avg_cuda_ms"],
        "arith_intensity": ai,
        "bound":         classify_bound(ai),
        "top_kernel":    (r["top_kernels"][0]["name"][:40] if r.get("top_kernels") else "N/A"),
    })

df = pd.DataFrame(rows).sort_values(["family","bits"], ascending=[True,False])
print(df[["description","bits","infer_ms","tok_per_sec","peak_gpu_gb","arith_intensity","bound"]].to_string(index=False))


## 2 · Profiler Summary Table (Report-Ready)

In [ ]:
# ── Latex-style HTML table ─────────────────────────────────────────────────
display_cols = {
    "description":   "Config",
    "bits":          "Bits",
    "infer_ms":      "Latency (ms)",
    "tok_per_sec":   "Tok/s",
    "peak_gpu_gb":   "Peak VRAM (GB)",
    "avg_cuda_ms":   "CUDA time (ms)",
    "arith_intensity": "Arith. Intensity",
    "bound":         "Bound classification",
    "top_kernel":    "Dominant kernel",
}

table_df = df[list(display_cols)].rename(columns=display_cols)
table_df = table_df.round({"Latency (ms)":1,"Tok/s":1,"Peak VRAM (GB)":2,
                            "CUDA time (ms)":1,"Arith. Intensity":1})

# Color code memory-bound red, compute-bound green
def color_bound(val):
    if "Compute" in str(val): return "background-color: #c8e6c9"
    if "Memory"  in str(val): return "background-color: #ffcdd2"
    if "Near"    in str(val): return "background-color: #fff9c4"
    return ""

styled = table_df.style.applymap(color_bound, subset=["Bound classification"])
display(styled)

# Also save as CSV for Team C's Pareto analysis
csv_path = OUT_DIR / "profiler_summary_table.csv"
table_df.to_csv(csv_path, index=False)
print(f"Saved → {csv_path}")


## 3 · Compute-Bound vs. Memory-Bound Per Precision

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── LEFT: Arithmetic Intensity bars grouped by family ─────────────────────
families = df["family"].unique()
quant_order = ["fp16","gptq","awq","nf4"]
quant_labels = {"fp16":"FP16","gptq":"GPTQ INT4/8","awq":"AWQ INT4","nf4":"NF4"}
colors = {"fp16":"#2196F3","gptq":"#F44336","awq":"#4CAF50","nf4":"#FF9800"}

ax1 = axes[0]
n_families = len(families)
x_base = np.arange(len(quant_order))
bar_w  = 0.22
offsets = np.linspace(-(n_families-1)*bar_w/2, (n_families-1)*bar_w/2, n_families)

for fi, (family, offset) in enumerate(zip(families, offsets)):
    fam_df = df[df["family"]==family]
    ai_vals = []
    for qt in quant_order:
        sub = fam_df[fam_df["quant_type"]==qt]
        ai_vals.append(sub["arith_intensity"].mean() if not sub.empty else 0)
    bars = ax1.bar(x_base + offset, ai_vals, bar_w*0.9,
                   label=family, alpha=0.85)

ax1.axhline(A100_RIDGE, color="red", linestyle="--", lw=1.5, label=f"Ridge point ({A100_RIDGE:.0f})")
ax1.set_xticks(x_base)
ax1.set_xticklabels([quant_labels.get(q, q) for q in quant_order])
ax1.set_ylabel("Arithmetic Intensity (FLOPs / byte)")
ax1.set_title("Arithmetic Intensity per Quantization Method")
ax1.legend(fontsize=8); ax1.grid(axis="y", alpha=0.3)

# Annotation: above ridge = compute-bound, below = memory-bound
ax1.text(0.98, 0.98, "Above dashed line = Compute-bound",
         transform=ax1.transAxes, ha="right", va="top", fontsize=8, color="red")

# ── RIGHT: VRAM usage ──────────────────────────────────────────────────────
ax2 = axes[1]
for fi, (family, offset) in enumerate(zip(families, offsets)):
    fam_df = df[df["family"]==family]
    vram_vals = []
    for qt in quant_order:
        sub = fam_df[fam_df["quant_type"]==qt]
        vram_vals.append(sub["peak_gpu_gb"].mean() if not sub.empty else 0)
    ax2.bar(x_base + offset, vram_vals, bar_w*0.9, label=family, alpha=0.85)

ax2.set_xticks(x_base)
ax2.set_xticklabels([quant_labels.get(q, q) for q in quant_order])
ax2.set_ylabel("Peak VRAM (GB)")
ax2.set_title("Peak GPU Memory Usage per Quantization Method")
ax2.legend(fontsize=8); ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Cross-Model Systems Analysis — A100-80GB", fontsize=13, fontweight="bold")
plt.tight_layout()
p = OUT_DIR / "fig_compute_memory_bound.png"
fig.savefig(p, dpi=180, bbox_inches="tight"); plt.show()
print(f"Saved → {p}")


In [ ]:
# ── Per-precision detailed classification ─────────────────────────────────
print("\n" + "="*72)
print("MEMORY-BOUND vs COMPUTE-BOUND PER PRECISION — CROSS-MODEL ANALYSIS")
print("="*72)

for bits_val in [16, 8, 4]:
    sub = df[df["bits"]==bits_val][["description","family","quant_type","arith_intensity","bound"]]
    if sub.empty: continue
    print(f"\n  {bits_val}-bit ({len(sub)} configs):")
    for _, row in sub.iterrows():
        marker = "🔴" if "Memory" in row["bound"] else ("🟢" if "Compute" in row["bound"] else "🟡")
        print(f"    {marker} {row['description']:<30}  AI={row['arith_intensity']:>7.1f}  →  {row['bound']}")

print()
print("KEY FINDINGS:")
print("  • FP16 configs are universally memory-bound regardless of model family")
print(f"    (AI ≈ 50–80 FLOPs/byte  <<  ridge {A100_RIDGE:.0f})")
print("  • GPTQ INT4 with dequant+matmul overhead has higher AI than expected for 4-bit")
print("  • AWQ fused GEMV kernel has the lowest AI: very memory-bandwidth-limited")
print("  • NF4 (bitsandbytes) is constrained by the naive gemv_4bit scalar kernel")


## 4 · vLLM Throughput Comparison (Mistral-7B)

In [ ]:
if not vllm_results:
    print("vLLM results not loaded — skipping this section")
else:
    vllm_rows = []
    for key, r in vllm_results.items():
        vllm_rows.append({
            "config":  key,
            "kernel":  r.get("kernel","?"),
            "tok_s":   r["avg_tok_per_sec"],
            "mem_gb":  r["peak_gpu_mem_gb"],
        })
    vdf = pd.DataFrame(vllm_rows).sort_values("tok_s", ascending=False)

    fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(14, 5))

    bar_colors = {
        "cuBLAS":                "#2196F3",
        "gptq_marlin":           "#4CAF50",
        "gptq (exllama)":        "#FF7043",
        "fused_GEMV (AWQ)":      "#AB47BC",
        "bitsandbytes_HF_batched":"#FF9800",
    }

    # Throughput bars
    bars = ax3a.barh(vdf["config"], vdf["tok_s"],
                     color=[bar_colors.get(k,"gray") for k in vdf["kernel"]],
                     edgecolor="black", linewidth=0.5)
    for bar, (_, row) in zip(bars, vdf.iterrows()):
        ax3a.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
                  f"{row['tok_s']:.0f} tok/s  [{row['kernel']}]",
                  va="center", fontsize=8.5)
    ax3a.set_xlabel("Throughput (tokens / second)")
    ax3a.set_title("vLLM Throughput — Mistral-7B")
    ax3a.grid(axis="x", alpha=0.3)

    # Memory bars
    ax3b.barh(vdf["config"], vdf["mem_gb"],
              color=[bar_colors.get(k,"gray") for k in vdf["kernel"]],
              edgecolor="black", linewidth=0.5)
    ax3b.set_xlabel("Peak VRAM (GB, via nvidia-smi)")
    ax3b.set_title("vLLM Peak GPU Memory — Mistral-7B")
    ax3b.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    p3 = OUT_DIR / "fig_vllm_throughput.png"
    fig3.savefig(p3, dpi=180, bbox_inches="tight"); plt.show()
    print(f"Saved → {p3}")

    # ── Speedup table relative to FP16 ────────────────────────────────────
    fp16_tps = vdf[vdf["config"]=="mistral-7b-fp16"]["tok_s"].values[0] if "mistral-7b-fp16" in vdf["config"].values else None
    if fp16_tps:
        vdf["speedup_vs_fp16"] = (vdf["tok_s"] / fp16_tps).round(2)
        fp16_mem = vdf[vdf["config"]=="mistral-7b-fp16"]["mem_gb"].values[0]
        vdf["mem_saving_x"] = (fp16_mem / vdf["mem_gb"]).round(2)
        print("\nvLLM Summary (Mistral-7B):")
        print(vdf[["config","kernel","tok_s","speedup_vs_fp16","mem_gb","mem_saving_x"]].to_string(index=False))
        print("\n💡 Note: GPTQ INT4 Marlin achieves {:.1f}× throughput vs FP16 with {:.1f}× less VRAM".format(
            vdf[vdf["config"]=="mistral-7b-gptq-int4"]["speedup_vs_fp16"].values[0],
            vdf[vdf["config"]=="mistral-7b-gptq-int4"]["mem_saving_x"].values[0],
        ))


## 5 · Memory Efficiency Analysis

In [ ]:
# For each model family: how much VRAM does each quantization save vs FP16?

fig4, ax4 = plt.subplots(figsize=(12, 5))

families  = df["family"].unique()
quants    = ["gptq-int8","gptq-int4","awq-int4","nf4"]   # vs fp16 baseline

x       = np.arange(len(quants))
bar_w   = 0.25
offsets = np.linspace(-(len(families)-1)*bar_w/2,
                       (len(families)-1)*bar_w/2, len(families))
fam_colors = ["#1565C0","#2E7D32","#B71C1C"]

for fi, (family, offset) in enumerate(zip(families, offsets)):
    fam_df = df[df["family"]==family]
    fp16_row = fam_df[fam_df["quant_type"]=="fp16"]
    if fp16_row.empty: continue
    fp16_mem = fp16_row["peak_gpu_gb"].values[0]

    reductions = []
    for q in quants:
        qt, bits = q.split("-") if "-" in q else (q, "4")
        bit_int  = int(bits.replace("int",""))
        qrow = fam_df[(fam_df["quant_type"]==qt) & (fam_df["bits"]==bit_int)]
        if qrow.empty:
            qrow = fam_df[fam_df["quant_type"]==qt]
        reduction_pct = ((fp16_mem - qrow["peak_gpu_gb"].mean()) / fp16_mem * 100
                          if not qrow.empty else 0)
        reductions.append(max(0, reduction_pct))

    ax4.bar(x + offset, reductions, bar_w*0.9,
            label=family, color=fam_colors[fi], alpha=0.82, edgecolor="k", lw=0.5)

ax4.set_xticks(x)
ax4.set_xticklabels(["GPTQ INT8","GPTQ INT4","AWQ INT4","NF4"], fontsize=10)
ax4.set_ylabel("VRAM reduction vs FP16 baseline (%)")
ax4.set_title("GPU Memory Savings per Quantization Method (relative to FP16)",
              fontsize=11, fontweight="bold")
ax4.legend(fontsize=9); ax4.grid(axis="y", alpha=0.3)
ax4.set_ylim(0, 100)

plt.tight_layout()
p4 = OUT_DIR / "fig_memory_efficiency.png"
fig4.savefig(p4, dpi=180, bbox_inches="tight"); plt.show()
print(f"Saved → {p4}")


## 6 · Accuracy vs. Efficiency Scatter (Mistral-7B)

In [ ]:
# Combines calibration metrics (ECE, accuracy) with profiler efficiency (tok/s)
# for Mistral-7B.  This feeds into Team C's Pareto analysis.

if not cal_results:
    print("Calibration results not loaded — skipping scatter")
else:
    scatter_rows = []
    for ckey, datasets in cal_results.items():
        for dname, d in datasets.items():
            prow = df[df["config_key"]==ckey]
            tps  = prow["tok_per_sec"].values[0] if not prow.empty else None
            mem  = prow["peak_gpu_gb"].values[0] if not prow.empty else None
            scatter_rows.append({
                "config":   ckey,
                "dataset":  d["dataset"],
                "accuracy": d["accuracy"],
                "ECE":      d["ECE"],
                "tok_s_profiler": tps,
                "peak_gpu_gb":    mem,
            })

    sdf = pd.DataFrame(scatter_rows)

    fig5, axes5 = plt.subplots(1, 2, figsize=(15, 6))

    marker_map = {
        "mistral-7b-fp16":"o","mistral-7b-gptq-int8":"s",
        "mistral-7b-gptq-int4":"^","mistral-7b-awq-int4":"D",
        "mistral-7b-nf4":"P",
    }
    ds_colors = {"hellaswag":"#2196F3","triviaqa":"#4CAF50","pubmedqa":"#F44336"}

    for ax5, (xcol, xlabel) in zip(
        axes5,
        [("tok_s_profiler","Throughput (tok/s, PyTorch Profiler)"),
         ("peak_gpu_gb","Peak GPU Memory (GB)")]
    ):
        for ckey in sdf["config"].unique():
            sub = sdf[sdf["config"]==ckey]
            for _, row in sub.iterrows():
                ax5.scatter(
                    row[xcol], row["ECE"],
                    marker=marker_map.get(ckey,"o"),
                    color=ds_colors.get(row["dataset"],"gray"),
                    s=80, edgecolors="black", lw=0.5, alpha=0.85,
                )
                if xcol == "tok_s_profiler":
                    ax5.annotate(
                        f"{ckey.replace('mistral-7b-','')}
({row['dataset'][:5]})",
                        xy=(row[xcol], row["ECE"]),
                        xytext=(4, 2), textcoords="offset points", fontsize=6.5,
                    )
        ax5.set_xlabel(xlabel, fontsize=10)
        ax5.set_ylabel("Expected Calibration Error (ECE — lower is better)", fontsize=10)
        ax5.set_title(f"ECE vs. {xlabel.split('(')[0].strip()}")
        ax5.grid(alpha=0.3)
        handles = [mpatches.Patch(color=c, label=d) for d, c in ds_colors.items()]
        ax5.legend(handles=handles, title="Dataset", fontsize=8)

    plt.suptitle("Calibration Quality vs. Serving Efficiency — Mistral-7B", fontsize=12, fontweight="bold")
    plt.tight_layout()
    p5 = OUT_DIR / "fig_ece_vs_efficiency.png"
    fig5.savefig(p5, dpi=180, bbox_inches="tight"); plt.show()
    print(f"Saved → {p5}")


## 7 · Report Narrative Synthesis

In [ ]:
# ── Auto-generate key findings for the report profiling section ─────────────

print("=" * 72)
print(" TEAM B — WEEK 5 SYSTEMS SECTION: KEY FINDINGS")
print("=" * 72)

# FP16 baseline
fp16_rows = df[df["quant_type"]=="fp16"]
print("\n▶  FP16 Baselines (memory-bound on all model families):")
for _, r in fp16_rows.iterrows():
    print(f"     {r['description']:<30}  {r['tok_per_sec']:>6.1f} tok/s  "
          f"|  {r['peak_gpu_gb']:>5.1f} GB  |  AI={r['arith_intensity']:>5.1f}")

# Quantization efficiency
print("\n▶  4-bit configs vs FP16 baseline (per family):")
for fam in df["family"].unique():
    fam_df = df[df["family"]==fam]
    fp16   = fam_df[fam_df["quant_type"]=="fp16"]
    if fp16.empty: continue
    fp16_tps = fp16["tok_per_sec"].values[0]
    fp16_mem = fp16["peak_gpu_gb"].values[0]
    for qt in ["gptq","awq","nf4"]:
        row = fam_df[(fam_df["quant_type"]==qt) & (fam_df["bits"]==4)]
        if row.empty: continue
        tps = row["tok_per_sec"].values[0]
        mem = row["peak_gpu_gb"].values[0]
        speedup  = tps  / fp16_tps
        mem_save = (fp16_mem - mem) / fp16_mem * 100
        print(f"     {fam} {qt}-int4:  {speedup:.2f}× tok/s  |  "
              f"{mem_save:.0f}% VRAM saved  |  {classify_bound(row['arith_intensity'].values[0])}")

# Dominant kernels
print("\n▶  Dominant kernel per quant type (Mistral-7B, representative):")
quant_kernel = {}
for _, r in df[df["family"]=="mistral-7b"].iterrows():
    quant_kernel[r["quant_type"]] = r["top_kernel"]
for qt, kern in quant_kernel.items():
    print(f"     {qt:<8}  {kern}")

print("\n" + "-"*72)
print("See figures saved to:", OUT_DIR)
print("-"*72)


In [ ]:
# ── Save all figures list as JSON manifest for the report ────────────────
import datetime
manifest = {
    "generated": datetime.datetime.now().isoformat(),
    "team":      "Team B (Systems Lead)",
    "week":      5,
    "figures": {
        "fig_compute_memory_bound.png": "Arithmetic intensity per quant method, grouped by model family",
        "fig_vllm_throughput.png":      "vLLM throughput and peak memory (Mistral-7B)",
        "fig_memory_efficiency.png":    "VRAM reduction vs FP16 baseline",
        "fig_ece_vs_efficiency.png":    "ECE vs throughput/memory (Mistral-7B calibration overlay)",
        "crossmodel_roofline.png":      "Cross-model Roofline plot (all 15 configs, from Notebook 1)",
        "crossmodel_kernel_heatmap.png":"Top-kernel CUDA time heatmap (all 15 configs, from Notebook 1)",
    },
    "outputs_for_team_c": ["profiler_summary_table.csv"],
}
with open(OUT_DIR / "week5_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("Manifest saved. Week 5 systems analysis complete.")
